In [2]:

# ============================================================
# 1. IMPORTAR BIBLIOTECAS E MONTAR O GOOGLE DRIVE
# ============================================================

from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')


# ============================================================
# 2. DEFINIR OS CAMINHOS
# ============================================================

# Caminho do arquivo CNV original
caminho_cnv = '/content/drive/MyDrive/datasus/mun/MUNICBR.CNV'

# Pasta onde o arquivo CSV será salvo
pasta_saida = '/content/drive/MyDrive/datasus/mun/'

# Nome do arquivo CSV de saída
nome_csv = 'MUNICBR_transcrito.csv'

# Caminho completo do CSV
caminho_csv = os.path.join(pasta_saida, nome_csv)


# ============================================================
# 3. VERIFICAR OS CAMINHOS
# ============================================================

if not os.path.exists(caminho_cnv):
    raise FileNotFoundError(
        f"Arquivo CNV não encontrado: {caminho_cnv}"
    )

os.makedirs(pasta_saida, exist_ok=True)


# ============================================================
# 4. FUNÇÃO DE TRANSCRIÇÃO DO MUNICBR
# ============================================================

def transcrever_municbr(caminho_arquivo, caminho_saida):

    registros = []

    with open(caminho_arquivo, 'r', encoding='latin1') as arquivo:
        linhas = arquivo.readlines()

    # Cabeçalho do arquivo CNV
    cabecalho = linhas[0].strip().split()
    quantidade_esperada = int(cabecalho[0])

    for numero_linha, linha in enumerate(linhas[1:], start=2):

        linha = linha.rstrip('\r\n')

        if not linha.strip():
            continue

        if len(linha) < 66:
            raise ValueError(
                f"Linha {numero_linha} possui estrutura inesperada: {linha!r}"
            )

        # Extração dos campos
        ordem = linha[0:7].strip()
        codigo = linha[9:15].strip()
        descricao = linha[16:57].strip()

        # Preservar o campo original de códigos
        campo_original = linha[60:].strip()

        # Separar os códigos adicionais
        lista_codigos = campo_original.split(',')
        codigos_adicionais = lista_codigos[1:]

        registros.append({
            'ordem': ordem,
            'codigo': codigo,
            'descricao': descricao,
            'campo_codigos_original': campo_original,
            'codigos_adicionais': ';'.join(codigos_adicionais)
        })

    df = pd.DataFrame(registros)

    # ========================================================
    # 5. VALIDAÇÕES
    # ========================================================

    if len(df) != quantidade_esperada:
        raise ValueError(
            f"Quantidade divergente! Esperado: {quantidade_esperada}; "
            f"Encontrado: {len(df)}"
        )

    if not df['codigo'].str.fullmatch(r'\d{6}').all():
        raise ValueError(
            "Foram encontrados códigos principais inválidos."
        )

    if df['descricao'].eq('').any():
        raise ValueError(
            "Foram encontradas descrições vazias."
        )

    # ========================================================
    # 6. EXPORTAR PARA CSV
    # ========================================================

    df.to_csv(
        caminho_saida,
        index=False,
        encoding='utf-8-sig'
    )

    print('Transcrição concluída!')
    print(f'Total de registros: {len(df)}')
    print(f'Arquivo salvo em: {caminho_saida}')

    return df


# ============================================================
# 7. EXECUTAR A TRANSCRIÇÃO
# ============================================================

df_municbr = transcrever_municbr(
    caminho_cnv,
    caminho_csv
)


# ============================================================
# 8. VISUALIZAR RESULTADO
# ============================================================

display(df_municbr.head(10))


Mounted at /content/drive
Transcrição concluída!
Total de registros: 5598
Arquivo salvo em: /content/drive/MyDrive/datasus/mun/MUNICBR_transcrito.csv


,ordem,codigo,descricao,campo_codigos_original,codigos_adicionais
0,1,110001,Alta Floresta D'Oeste,110001,
1,2,110037,Alto Alegre dos Parecis,110037,
2,3,110040,Alto Paraíso,110040,
3,4,110034,Alvorada D'Oeste,110034,
4,5,110002,Ariquemes,110002,
5,6,110045,Buritis,110045,
6,7,110003,Cabixi,110003,
7,8,110060,Cacaulândia,110060,
8,9,110004,Cacoal,110004,
9,10,110070,Campo Novo de Rondônia,110070,
